In [11]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

import torch


In [12]:

env = gym.make(
    "LunarLander-v3", 
    continuous=False, 
    gravity= -10, 
    enable_wind=1, 
    wind_power=15, 
    turbulence_power=1.5)

In [13]:
model = PPO("MlpPolicy", env, verbose=1)

model.learn(total_timesteps=300000)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 86.6     |
|    ep_rew_mean     | -284     |
| time/              |          |
|    fps             | 6292     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 84.1        |
|    ep_rew_mean          | -247        |
| time/                   |             |
|    fps                  | 4931        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008803286 |
|    clip_fraction        | 0.0534      |
|    clip_range           | 0.2         |
|    entropy_loss   

In [14]:
import os
main_dir = os.getcwd()
model_dir = os.path.join(main_dir, "models/lunar_lander_v2")
video_dir = os.path.join(main_dir, "videos/lunar_lander_v2")

model.save(os.path.join(model_dir, "ppo_lunar_lander_v2"))
env.close()

In [15]:
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10, deterministic=True)

/opt/miniconda3/envs/pytorch_csiro/lib/python3.10/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


In [16]:
print(f"mean_reward: {mean_reward:.2f} of std. {std_reward:.2f}")

mean_reward: 21.76 of std. 109.63


In [17]:
from gymnasium.wrappers import RecordVideo


render_env = gym.make(
    "LunarLander-v3",
    continuous=False,
    gravity=-10,
    enable_wind=0,
    wind_power=15,
    turbulence_power=1.5,
    render_mode="rgb_array"
)

video_env = RecordVideo(
    render_env, 
    video_dir, 
    episode_trigger=lambda ep: True,
    name_prefix='v2_lander')


for ep in range(5):
    state, _ = video_env.reset()
    done = False
    total_reward = 0

    while not done:
        action,_ = model.predict(state, deterministic=True)
        state, reward, terminated, truncated, info = video_env.step(action)
        done = terminated or truncated
        total_reward += reward

    print(f"Episode {ep}, reward: {total_reward}")

video_env.close()


Episode 0, reward: 101.97829098317943
Episode 1, reward: 8.664626266517296
Episode 2, reward: 172.43264264246469
Episode 3, reward: 74.59567868822852
Episode 4, reward: 111.3469480351732


NameError: name 'gym' is not defined